# Conversational RAG with Redis (Starter)
This starter notebook verifies Redis, Chroma and OpenAI setup.

In [1]:
from dotenv import load_dotenv
load_dotenv()

import os, redis
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

r=redis.Redis(
    host=os.getenv("REDIS_HOST","localhost"),
    port=int(os.getenv("REDIS_PORT",6379)),
    db=int(os.getenv("REDIS_DB",0)),
    decode_responses=True
)
print("Redis:",r.ping())

Redis: True


In [2]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader

loader = DirectoryLoader(
    path="data",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

documents = loader.load()

print(f"Total Documents: {len(documents)}")

C:\Users\SHAILENDRA\AppData\Local\Temp\ipykernel_28660\271497775.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader


Total Documents: 3


In [3]:
documents[0]

Document(metadata={'source': 'data\\doc_0.txt'}, page_content='\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ')

In [4]:


print(documents[0].page_content[:500])


    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interacti


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"Original Documents : {len(documents)}")
print(f"Total Chunks       : {len(chunks)}")

Original Documents : 3
Total Chunks       : 7


In [6]:
chunks[0]

Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals')

In [7]:
print(chunks[0].page_content)

Machine Learning Fundamentals


In [8]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x000001C28FB0C210>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000001C28FBA9310>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [9]:
sample_text = "Python is a popular programming language."

vector = embeddings.embed_query(sample_text)

print(f"Vector Length : {len(vector)}")

Vector Length : 1536


In [11]:
persist_directory = "./chroma_db"

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="rag_collection"
)

print("Vector Store Created Successfully")

Vector Store Created Successfully


In [12]:
print(f"Total Vectors : {vectorstore._collection.count()}")

Total Vectors : 14


In [13]:
import os

os.listdir(persist_directory)

['c909b720-7b10-4428-81ac-0dc7a201aed8', 'chroma.sqlite3']

In [14]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001C291D69490>, search_kwargs={'k': 3})

In [17]:
query = "What is Deep Learning?"

retrieved_docs = retriever.invoke(query)

print(f"Retrieved Documents : {len(retrieved_docs), retrieved_docs}")

Retrieved Documents : (3, [Document(id='d0c80b68-fefd-4e62-902c-4e4f06a1b468', metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks'), Document(id='3d83a580-73e0-4470-8d79-0f79299ca2f9', metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks'), Document(id='9012f3b0-632d-4a8a-b6cc-7eaa7d3dd4bb', metadata={'source': 'data\\doc_1.txt'}, page_content='Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers')])


In [18]:
for i, doc in enumerate(retrieved_docs, start=1):
    print("=" * 80)
    print(f"Document {i}")
    print("=" * 80)
    print(doc.page_content)

Document 1
Deep Learning and Neural Networks
Document 2
Deep Learning and Neural Networks
Document 3
Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers


In [19]:
from langchain_core.prompts import ChatPromptTemplate

rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are a query rewriting assistant.

Given the conversation history and the latest user question,
rewrite the latest question into a standalone question.

Do NOT answer the question.

Return only the rewritten question.""",
        ),
        (
            "human",
            """Conversation History:
{chat_history}

Latest Question:
{question}""",
        ),
    ]
)

rewrite_prompt

ChatPromptTemplate(input_variables=['chat_history', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a query rewriting assistant.\n\nGiven the conversation history and the latest user question,\nrewrite the latest question into a standalone question.\n\nDo NOT answer the question.\n\nReturn only the rewritten question.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['chat_history', 'question'], input_types={}, partial_variables={}, template='Conversation History:\n{chat_history}\n\nLatest Question:\n{question}'), additional_kwargs={})])

In [20]:
rewrite_prompt.input_variables

['chat_history', 'question']

In [21]:
chat_history = """
Human: Tell me about Python.

AI: Python is a high-level programming language used for web development,
AI, automation, and data science.
"""

question = "What are its popular libraries?"

In [22]:
prompt_value = rewrite_prompt.invoke(
    {
        "chat_history": chat_history,
        "question": question,
    }
)

print(prompt_value)

messages=[SystemMessage(content='You are a query rewriting assistant.\n\nGiven the conversation history and the latest user question,\nrewrite the latest question into a standalone question.\n\nDo NOT answer the question.\n\nReturn only the rewritten question.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Conversation History:\n\nHuman: Tell me about Python.\n\nAI: Python is a high-level programming language used for web development,\nAI, automation, and data science.\n\n\nLatest Question:\nWhat are its popular libraries?', additional_kwargs={}, response_metadata={})]


In [24]:
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

In [25]:
from langchain_core.output_parsers import StrOutputParser

rewrite_chain = (
    rewrite_prompt
    | llm
    | StrOutputParser()
)

rewrite_chain

ChatPromptTemplate(input_variables=['chat_history', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a query rewriting assistant.\n\nGiven the conversation history and the latest user question,\nrewrite the latest question into a standalone question.\n\nDo NOT answer the question.\n\nReturn only the rewritten question.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['chat_history', 'question'], input_types={}, partial_variables={}, template='Conversation History:\n{chat_history}\n\nLatest Question:\n{question}'), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.14', 'langchain-openai': '1.3.5'}}, output_version=None, profile={'name': 'GPT-4.1 mini', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens'

In [26]:
standalone_question = rewrite_chain.invoke(
    {
        "chat_history": chat_history,
        "question": question,
    }
)

print(standalone_question)

What are the popular libraries in Python?


In [28]:
from langchain_core.prompts import ChatPromptTemplate

answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are a helpful AI assistant.

Answer the user's question ONLY using the provided context.

If the answer is not available in the context, simply reply:

"I don't know."

Context:
{context}
""",
        ),
        (
            "human",
            "{question}",
        ),
    ]
)

answer_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='You are a helpful AI assistant.\n\nAnswer the user\'s question ONLY using the provided context.\n\nIf the answer is not available in the context, simply reply:\n\n"I don\'t know."\n\nContext:\n{context}\n'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})])

In [29]:
from langchain_core.output_parsers import StrOutputParser

answer_chain = (
    answer_prompt
    | llm
    | StrOutputParser()
)

answer_chain

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='You are a helpful AI assistant.\n\nAnswer the user\'s question ONLY using the provided context.\n\nIf the answer is not available in the context, simply reply:\n\n"I don\'t know."\n\nContext:\n{context}\n'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.14', 'langchain-openai': '1.3.5'}}, output_version=None, profile={'name': 'GPT-4.1 mini', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_input

In [30]:
import uuid

session_id = str(uuid.uuid4())

print(session_id)

6a349837-6a93-4678-96e0-4026dcea5eab


In [84]:
import json
from datetime import datetime

def save_message(session_id, role, content):
    key = f"chat:{session_id}"

    message = {
        "role": role,
        "content": content,
        "timestamp": datetime.utcnow().isoformat()
    }

    r.rpush(key, json.dumps(message))

    # Refresh TTL on every new message
    r.expire(key, 60 * 60 * 24 * 7)

In [32]:
def load_messages(session_id):
    key = f"chat:{session_id}"

    messages = r.lrange(key, 0, -1)

    return [json.loads(message) for message in messages]

In [33]:
def clear_session(session_id):
    key = f"chat:{session_id}"
    r.delete(key)

In [41]:
save_message(
    session_id,
    "human",
    "Tell me about Python"
)

save_message(
    session_id,
    "ai",
    "Python is a programming language."
)

In [42]:
conversation = load_messages(session_id)

conversation

[{'role': 'human', 'content': 'Tell me about Python'},
 {'role': 'ai', 'content': 'Python is a programming language.'}]

In [36]:
r.lrange(f"chat:{session_id}", 0, -1)

['{"role": "human", "content": "Tell me about Python"}',
 '{"role": "ai", "content": "Python is a programming language."}']

In [ ]:
# clear_session(session_id)

load_messages(session_id)

[]

In [38]:
from langchain_core.messages import (
    HumanMessage,
    AIMessage,
)

In [39]:
def convert_to_langchain_messages(messages):

    chat_history = []

    for message in messages:

        if message["role"] == "human":
            chat_history.append(
                HumanMessage(content=message["content"])
            )

        elif message["role"] == "ai":
            chat_history.append(
                AIMessage(content=message["content"])
            )

    return chat_history

In [43]:
conversation = load_messages(session_id)

conversation

[{'role': 'human', 'content': 'Tell me about Python'},
 {'role': 'ai', 'content': 'Python is a programming language.'}]

In [44]:
chat_history = convert_to_langchain_messages(conversation)

chat_history

[HumanMessage(content='Tell me about Python', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Python is a programming language.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [45]:
chat_history_text = "\n".join(
    [
        f"{message['role'].capitalize()}: {message['content']}"
        for message in conversation
    ]
)

print(chat_history_text)

Human: Tell me about Python
Ai: Python is a programming language.


In [46]:
conversation = load_messages(session_id)

conversation

[{'role': 'human', 'content': 'Tell me about Python'},
 {'role': 'ai', 'content': 'Python is a programming language.'}]

In [61]:
question = "NLP?"

print(question)

NLP?


In [62]:
standalone_question = rewrite_chain.invoke(
    {
        "chat_history": chat_history,
        "question": question,
    }
)
print(chat_history)
print(question)
print(standalone_question)

[HumanMessage(content='Tell me about Python', additional_kwargs={}, response_metadata={}), AIMessage(content='Python is a programming language.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
NLP?
What is NLP?


In [54]:
retrieved_docs = retriever.invoke(standalone_question)

print(f"Retrieved {len(retrieved_docs)} documents")

Retrieved 3 documents


In [55]:
print("Using Original Question")
print("-" * 80)

original_docs = retriever.invoke(question)

for doc in original_docs:
    print(doc.metadata["source"])

print("\nUsing Rewritten Question")
print("-" * 80)

rewritten_docs = retriever.invoke(standalone_question)

for doc in rewritten_docs:
    print(doc.metadata["source"])

Using Original Question
--------------------------------------------------------------------------------
data\doc_1.txt
data\doc_1.txt
data\doc_1.txt

Using Rewritten Question
--------------------------------------------------------------------------------
data\doc_1.txt
data\doc_1.txt
data\doc_2.txt


In [56]:
context = "\n\n".join(
    doc.page_content
    for doc in retrieved_docs
)

print(context)

Deep Learning and Neural Networks

Deep Learning and Neural Networks

Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.


In [63]:
answer = answer_chain.invoke(
    {
        "context": context,
        "question": standalone_question,
    }
)
print(context)
print(standalone_question)
print(answer)

Deep Learning and Neural Networks

Deep Learning and Neural Networks

Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
What is NLP?
NLP, or Natural Language Processing, is a field of AI that focuses on the interaction between computers and human language.


In [64]:
save_message(
    session_id=session_id,
    role="human",
    content=question
)

In [65]:
save_message(
    session_id=session_id,
    role="ai",
    content=answer
)

In [66]:
conversation = load_messages(session_id)

conversation

[{'role': 'human', 'content': 'Tell me about Python'},
 {'role': 'ai', 'content': 'Python is a programming language.'},
 {'role': 'human', 'content': 'NLP?'},
 {'role': 'ai',
  'content': 'NLP, or Natural Language Processing, is a field of AI that focuses on the interaction between computers and human language.'}]

### Create one function that executes the complete Conversational RAG pipeline.

In [71]:
def ask(session_id, question):

    # Step 1 - Load Conversation
    conversation = load_messages(session_id)

    # Step 2 - Build Chat History
    chat_history = "\n".join(
        f"{message['role'].capitalize()}: {message['content']}"
        for message in conversation
    )

    # Step 3 - Rewrite Question
    standalone_question = rewrite_chain.invoke(
        {
            "chat_history": chat_history,
            "question": question,
        }
    )

    # Step 4 - Retrieve Documents
    retrieved_docs = retriever.invoke(standalone_question)

    # Step 5 - Build Context
    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )

    # Step 6 - Generate Answer
    answer = answer_chain.invoke(
        {
            "context": context,
            "question": standalone_question,
        }
    )

    # Step 7 - Save User Message
    save_message(
        session_id=session_id,
        role="human",
        content=question,
    )

    # Step 8 - Save AI Response
    save_message(
        session_id=session_id,
        role="ai",
        content=answer,
    )

    return answer

### Conversation 1

In [73]:
session_id = "demo-session"

clear_session(session_id)

response = ask(
    session_id,
    "Tell NLP."
)

print(response)

NLP, or Natural Language Processing, is a field of AI that focuses on the interaction between computers and human language.


In [75]:
response = ask(
    session_id,
    "What its use case" #its followup question 
)

print(response)

The use cases of Natural Language Processing (NLP) include text classification, named entity recognition, sentiment analysis, machine translation, and question answering.


In [76]:
response = ask(
    session_id,
    "Which is sentiment analysis?"
)

print(response)

Sentiment analysis in the context of Natural Language Processing (NLP) is a key task that involves determining the sentiment or emotional tone expressed in a piece of text. It helps in understanding whether the text conveys positive, negative, or neutral feelings.


### Verify conversation in redis 

In [77]:
conversation = load_messages(session_id)

for message in conversation:
    print(f"{message['role'].upper():<6}: {message['content']}")
    print()

HUMAN : Tell NLP.

AI    : NLP, or Natural Language Processing, is a field of AI that focuses on the interaction between computers and human language.

HUMAN : What its use case

AI    : The use cases of Natural Language Processing (NLP) include text classification, named entity recognition, sentiment analysis, machine translation, and question answering.

HUMAN : What its use case

AI    : The use cases of Natural Language Processing (NLP) include text classification, named entity recognition, sentiment analysis, machine translation, and question answering.

HUMAN : Which is sentiment analysis?

AI    : Sentiment analysis in the context of Natural Language Processing (NLP) is a key task that involves determining the sentiment or emotional tone expressed in a piece of text. It helps in understanding whether the text conveys positive, negative, or neutral feelings.



## Debug

In [79]:
def ask_debug(session_id, question):

    # Load conversation
    conversation = load_messages(session_id)

    # Build history
    chat_history = "\n".join(
        f"{m['role'].capitalize()}: {m['content']}"
        for m in conversation
    )

    # Rewrite
    standalone_question = rewrite_chain.invoke(
        {
            "chat_history": chat_history,
            "question": question,
        }
    )

    # Retrieve
    retrieved_docs = retriever.invoke(standalone_question)

    # Build Context
    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )

    # Answer
    answer = answer_chain.invoke(
        {
            "context": context,
            "question": standalone_question,
        }
    )

    # Save conversation
    save_message(session_id, "human", question)
    save_message(session_id, "ai", answer)

    return {
        "original_question": question,
        "standalone_question": standalone_question,
        "chat_history": chat_history,
        "documents": retrieved_docs,
        "context": context,
        "answer": answer,
    }

In [80]:
result = ask_debug(
    "demo",
    "Tell NLP"
)

In [81]:
result.keys()

dict_keys(['original_question', 'standalone_question', 'chat_history', 'documents', 'context', 'answer'])

In [82]:
print(result)

{'original_question': 'Tell NLP', 'standalone_question': 'What is NLP?', 'chat_history': '', 'documents': [Document(id='b26a1e5c-d305-4172-b672-2f7728c2449f', metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'), Document(id='17821c48-f5ac-4423-971d-2fe498874337', metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognit

# Matrics

In [87]:
from rag_eval.metrics_collector import MetricsCollector

metrics = MetricsCollector()

In [88]:
import time

metrics.start("retrieval")

time.sleep(0.35)

metrics.stop("retrieval")

0.3511884000035934

In [89]:
metrics.set("question", "What is Python?")

metrics.set("model", "gpt-4.1-mini")

metrics.set("embedding_model", "text-embedding-3-small")

In [90]:
metrics.increment("documents")

metrics.increment("documents")

metrics.increment("documents")

In [91]:
metrics.report()

RAG METRICS REPORT

Metadata
question                 : What is Python?
model                    : gpt-4.1-mini
embedding_model          : text-embedding-3-small

Timers
retrieval                : 351.19 ms

Counters
documents                : 3


In [93]:
metrics.start("llm")

answer = answer_chain.invoke(
        {
            "context": context,
            "question": standalone_question,
        }
    )

metrics.stop("llm")

5.040969000023324